<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта: Система строк заказов</h2>

----

### Вариант задания  15


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Крутые линии заказов для какой-нибудь таблицы.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте полиморфизм с перекрытием и прегегрузкой методов, а также generic классы

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
public interface ITrackable
{
    string TrackingNumber { get; set; }
    void UpdateTracking(string status);
    void UpdateTracking(string status, string location);
}

public interface IReturnable
{
    int ReturnWindowDays { get; set; }
    bool RequestReturn(string reason);
}

class PackageSize
{
    public decimal Size { get; private set; }
    public decimal X1 { get; set; }
    public decimal Y1 { get; set; }
    public decimal Z1 { get; set; }
    public decimal X2 { get; set; }
    public decimal Y2 { get; set; }
    public decimal Z2 { get; set; }

    public PackageSize(decimal x1, decimal y1, decimal z1,
        decimal x2, decimal y2, decimal z2)
    {
        X1 = x1;
        Y1 = y1;
        Z1 = z1;
        X2 = x2;
        Y2 = y2;
        Z2 = z2;
        Size = Math.Abs((x2 - x1) * (y2 - y1) * (z2 - z1));
    }
}

class OrderLine
{
    public Guid ProductId { get; private set; } = Guid.NewGuid();
    public string ProductName { get; set; }
    public decimal Price { get; protected set; }
    public PackageSize Box { get; private set; } = new PackageSize(1, 1, 1, 2, 2, 2);
    public decimal Weight { get; set; }
    public bool IsFragile { get; set; }
    public string Category { get; set; }
    public string OriginCountry { get; set; }
    public DateTime CreatedDate { get; set; }
    public int PriorityLevel { get; set; }
    public bool IsExpressDelivery { get; set; }

    public OrderLine(string productName, decimal price)
    {
        ProductName = productName;
        Price = price;
        Weight = 1.0M;
        IsFragile = false;
        Category = "Общая";
        OriginCountry = "Не указана";
        CreatedDate = DateTime.Now;
        PriorityLevel = 0;
        IsExpressDelivery = false;
    }

    public virtual decimal CalculateTotal()
    {
        return Price * 1.25M * 1.01M + 10;
    }

    public virtual decimal CalculateTotal(decimal discountPercent)
    {
        return (Price * 1.25M * 1.01M + 10) * (1M - discountPercent / 100M);
    }

    public virtual void UpdatePrice(decimal newPrice)
    {
        Price = newPrice;
    }

    public void UpdatePrice(decimal newPrice, decimal taxMultiplier)
    {
        Price = newPrice * taxMultiplier;
    }

    public void ChangeSizeBox(PackageSize box)
    {
        Box = box;
    }

    public virtual decimal GetShippingCost()
    {
        return IsExpressDelivery ? Weight * 10.0M + 25.0M : Weight * 5.0M;
    }

    public virtual decimal GetShippingCost(bool isExpress)
    {
        return isExpress ? Weight * 10.0M + 25.0M : Weight * 5.0M;
    }

    public virtual void ApplyQualityCheck()
    {
        Console.WriteLine($"Базовая проверка качества для товара: {ProductName}. Проверка пройдена.");
    }

    public void PrintWarning()
    {
        if (IsFragile)
        {
            Console.WriteLine($"Внимание: Товар {ProductName} хрупкий!");
        }
    }

    public string GetProductDetails()
    {
        return $"ID продукта: {ProductId}\n" +
            $"Название продукта: {ProductName}\n" +
            $"Себестоимость продукта: {Price}\n" +
            $"Размер упаковки для товара: {Box.Size} м^3\n" +
            $"Вес: {Weight} кг, Категория: {Category}\n" +
            $"Страна происхождения: {OriginCountry}, Приоритет: {PriorityLevel}\n";
    }
}

class StandardLine : OrderLine
{
    public int Units { get; set; }
    public string Supplier { get; set; }
    public DateTime RestockDate { get; set; }
    public int WarehouseZone { get; set; }
    public string QualityStatus { get; set; }

    public StandardLine(string productName, decimal price, int units)
        : base(productName, price)
    {
        Units = units;
        Supplier = "Неизвестен";
        RestockDate = DateTime.Now.AddDays(7);
        WarehouseZone = 1;
        QualityStatus = "Ожидает проверки";
    }

    public override decimal CalculateTotal()
    {
        return (Price * 1.25M * 1.01M + 10) * Units;
    }

    public override decimal CalculateTotal(decimal discountPercent)
    {
        return (Price * 1.25M * 1.01M + 10) * Units * (1M - discountPercent / 100M);
    }

    public override decimal GetShippingCost()
    {
        return IsExpressDelivery ? Weight * 8.0M * Units + 30.0M : base.GetShippingCost();
    }

    public override decimal GetShippingCost(bool isExpress)
    {
        return isExpress ? Weight * 8.0M * Units + 30.0M : base.GetShippingCost();
    }

    public override void ApplyQualityCheck()
    {
        QualityStatus = Units > 0 ? "Проверено на складе" : "Нет в наличии";
        Console.WriteLine($"Стандартная проверка партии товара {ProductName}: статус '{QualityStatus}'.");
    }

    public bool CheckAvailability()
    {
        return Units > 0;
    }

    public void ChangeUnits(int delta)
    {
        Units += delta;
        if (Units < 0) Units = 0;
    }
}

class BulkyStandardLine : StandardLine, ITrackable, IReturnable
{
    public decimal HandlingFee { get; set; }
    public string TrackingNumber { get; set; }
    public int ReturnWindowDays { get; set; }
    public int RequiredLoaders { get; set; }
    public string TransportType { get; set; }

    public BulkyStandardLine(string productName, decimal price, int units, decimal handlingFee)
        : base(productName, price, units)
    {
        HandlingFee = handlingFee;
        TrackingNumber = "-";
        ReturnWindowDays = 14;
        RequiredLoaders = 2;
        TransportType = "Стандартный грузовик";
    }

    public override decimal CalculateTotal()
    {
        return base.CalculateTotal() + HandlingFee;
    }

    public override decimal CalculateTotal(decimal discountPercent)
    {
        return base.CalculateTotal(discountPercent) + HandlingFee;
    }

    public override decimal GetShippingCost()
    {
        return base.GetShippingCost() + (IsExpressDelivery ? 100.0M : 50.0M);
    }

    public override decimal GetShippingCost(bool isExpress)
    {
        return isExpress ? base.GetShippingCost(true) + 100.0M : base.GetShippingCost() + 50.0M;
    }

    public override void ApplyQualityCheck()
    {
        Console.WriteLine($"Усиленная проверка крупногабаритного груза {ProductName} (грузчиков: {RequiredLoaders}, транспорт: {TransportType}).");
    }

    public void UpdateTracking(string status)
    {
        Console.WriteLine($"Статус доставки ({TrackingNumber}): {status}");
    }

    public void UpdateTracking(string status, string location)
    {
        Console.WriteLine($"Статус доставки ({TrackingNumber}): {status}, местоположение: {location}");
    }

    public bool RequestReturn(string reason)
    {
        Console.WriteLine($"Запрос на возврат {ProductName} по причине: {reason}");
        return true;
    }
}

class SpecialLine : OrderLine
{
    public int PercentDiscount { get; set; }
    public string PromoCode { get; set; }
    public DateTime ExpirationDate { get; set; }
    public decimal MinOrderAmount { get; set; }
    public string PromotionName { get; set; }

    public SpecialLine(string productName, decimal price, int percentDiscount)
        : base(productName, price)
    {
        PercentDiscount = percentDiscount;
        PromoCode = "ПРОМОКОД_ОТСУТСТВУЕТ";
        ExpirationDate = DateTime.Now.AddMonths(1);
        MinOrderAmount = 0M;
        PromotionName = "Обычная акция";
    }

    public override decimal CalculateTotal()
    {
        return (Price * 1.25M * 1.01M + 10) * (1M - PercentDiscount / 100M);
    }

    public override decimal CalculateTotal(decimal additionalDiscount)
    {
        decimal totalDiscount = PercentDiscount + additionalDiscount;
        if (totalDiscount > 90M) totalDiscount = 90M;
        return (Price * 1.25M * 1.01M + 10) * (1M - totalDiscount / 100M);
    }

    public override void UpdatePrice(decimal newPrice)
    {
        Price = newPrice * (1M - PercentDiscount / 100M);
    }

    public void ApplyPromo(string code)
    {
        if (code == PromoCode && DateTime.Now <= ExpirationDate)
        {
            PercentDiscount += 5;
        }
    }

    public void ApplyPromo(string code, int extraDiscount)
    {
        if (code == PromoCode && DateTime.Now <= ExpirationDate)
        {
            PercentDiscount += extraDiscount;
        }
    }

    public override void ApplyQualityCheck()
    {
        Console.WriteLine($"Проверка акционного товара {ProductName} по акции '{PromotionName}'. Скидка: {PercentDiscount}%.");
    }
}

class FreeLine : OrderLine, ITrackable
{
    public decimal Prepayment { get; set; }
    public string SponsorName { get; set; }
    public string TrackingNumber { get; set; }
    public string GrantNumber { get; set; }
    public bool RequiresDocumentation { get; set; }

    public FreeLine(string productName, decimal price, decimal prepayment)
        : base(productName, price)
    {
        Prepayment = prepayment;
        SponsorName = "Аноним";
        TrackingNumber = "Н/Д";
        GrantNumber = "БЕЗ_ГРАНТА";
        RequiresDocumentation = false;
    }

    public override decimal CalculateTotal()
    {
        decimal total = Price * 1.25M * 1.01M + 10 - Prepayment;
        return total < 0 ? 0 : total;
    }

    public override decimal CalculateTotal(decimal subsidy)
    {
        decimal total = Price * 1.25M * 1.01M + 10 - Prepayment - subsidy;
        return total < 0 ? 0 : total;
    }

    public void GetSponsorInfo()
    {
        Console.WriteLine($"Спонсор бесплатного товара: {SponsorName}");
    }

    public void UpdateTracking(string status)
    {
        Console.WriteLine($"Отслеживание ({TrackingNumber}): {status}");
    }

    public void UpdateTracking(string status, string location)
    {
        Console.WriteLine($"Отслеживание ({TrackingNumber}): {status}, локация: {location}");
    }

    public override void ApplyQualityCheck()
    {
        Console.WriteLine($"Проверка благотворительного груза {ProductName} от спонсора {SponsorName}. Грант: {GrantNumber}.");
    }
}

class Order<T> where T : OrderLine
{
    private List<T> lines;
    public string OrderNumber { get; set; }
    public decimal TotalOrderCost { get; private set; }

    public Order(string orderNumber)
    {
        OrderNumber = orderNumber;
        lines = new List<T>();
        TotalOrderCost = 0;
    }

    public void AddLine(T line)
    {
        lines.Add(line);
        TotalOrderCost += line.CalculateTotal() + line.GetShippingCost();
    }

    public void AddLine(T line, bool isExpress)
    {
        line.IsExpressDelivery = isExpress;
        lines.Add(line);
        TotalOrderCost += line.CalculateTotal() + line.GetShippingCost();
    }

    public void PrintReceipt()
    {
        Console.WriteLine($"=== ЧЕК ЗАКАЗА {OrderNumber} ===");
        foreach (var line in lines)
        {
            line.PrintWarning();
            Console.WriteLine(line.GetProductDetails());
            Console.WriteLine($"Стоимость строки (с доставкой): {line.CalculateTotal() + line.GetShippingCost()}\n");
        }
        Console.WriteLine($"ИТОГО К ОПЛАТЕ: {TotalOrderCost}");
        Console.WriteLine("================================");
    }
}

class BatchProcessor<T> where T : OrderLine
{
    private List<T> batchItems = new List<T>();

    public void AddItem(T item)
    {
        batchItems.Add(item);
    }

    public void ProcessAll()
    {
        Console.WriteLine("=== ПАКЕТНАЯ ПРОВЕРКА ПОЗИЦИЙ ===");
        foreach (var item in batchItems)
        {
            item.ApplyQualityCheck();
        }
        Console.WriteLine("================================");
    }
}

StandardLine standardItem = new StandardLine("Катушка медной проволоки", 1000, 3);
standardItem.Weight = 2.5M;
standardItem.OriginCountry = "Россия";
standardItem.PriorityLevel = 1;

SpecialLine specialItem = new SpecialLine("Бочки с метиламином", 500, 20);
specialItem.PromoCode = "ХИМИЯ2026";
specialItem.UpdatePrice(600);
specialItem.UpdatePrice(600, 1.1M);
specialItem.IsFragile = true;
specialItem.OriginCountry = "Германия";
specialItem.PromotionName = "Оптовая скидка";
specialItem.ApplyPromo("ХИМИЯ2026", 10);

FreeLine freeItem = new FreeLine("Клетка с обезьянами", 150, 120);
freeItem.TrackingNumber = "TRK987654321";
freeItem.SponsorName = "Фонд защиты животных";
freeItem.GrantNumber = "ГРАНТ-2026-ЭКО";
freeItem.OriginCountry = "Бразилия";

BulkyStandardLine bulkyItem = new BulkyStandardLine("Промышленный станок", 50000, 1, 1500);
bulkyItem.Weight = 500M;
bulkyItem.TrackingNumber = "BLK123456789";
bulkyItem.RequiredLoaders = 4;
bulkyItem.TransportType = "Грузовой автопоезд";
bulkyItem.OriginCountry = "Беларусь";

PackageSize newBox = new PackageSize(0, 0, 0, 3, 3, 3);
standardItem.ChangeSizeBox(newBox);

Order<OrderLine> generalOrder = new Order<OrderLine>("№ 001-ОБЩИЙ");
generalOrder.AddLine(standardItem);
generalOrder.AddLine(specialItem);
generalOrder.AddLine(freeItem);
generalOrder.AddLine(bulkyItem, true);
generalOrder.PrintReceipt();

Order<StandardLine> standardOrder = new Order<StandardLine>("№ 002-СТАНДАРТ");
standardOrder.AddLine(standardItem);

BatchProcessor<OrderLine> batch = new BatchProcessor<OrderLine>();
batch.AddItem(standardItem);
batch.AddItem(specialItem);
batch.AddItem(freeItem);
batch.AddItem(bulkyItem);
batch.ProcessAll();

freeItem.GetSponsorInfo();
freeItem.UpdateTracking("В пути");
freeItem.UpdateTracking("Прибыл в терминал", "Москва");

bulkyItem.UpdateTracking("Прибыл на склад");
bulkyItem.UpdateTracking("Передан в службу доставки", "Тюмень");
bulkyItem.RequestReturn("Не подошел по размеру");

=== ЧЕК ЗАКАЗА № 001-ОБЩИЙ ===
ID продукта: 148fd877-4300-4e0c-889c-a9ef5784d444
Название продукта: Катушка медной проволоки
Себестоимость продукта: 1000
Размер упаковки для товара: 27 м^3
Вес: 2.5 кг, Категория: Общая
Страна происхождения: Россия, Приоритет: 1

Стоимость строки (с доставкой): 3830.0000

Внимание: Товар Бочки с метиламином хрупкий!
ID продукта: 8a8e054c-8744-45bb-98a0-647c32358ca4
Название продукта: Бочки с метиламином
Себестоимость продукта: 660.0
Размер упаковки для товара: 1 м^3
Вес: 1.0 кг, Категория: Общая
Страна происхождения: Германия, Приоритет: 0

Стоимость строки (с доставкой): 595.275000

ID продукта: c986b5ea-3453-4fd5-b96f-54d3c921f55c
Название продукта: Клетка с обезьянами
Себестоимость продукта: 150
Размер упаковки для товара: 1 м^3
Вес: 1.0 кг, Категория: Общая
Страна происхождения: Бразилия, Приоритет: 0

Стоимость строки (с доставкой): 84.3750

ID продукта: fb24f32a-9852-4044-adfe-b3e60465fca6
Название продукта: Промышленный станок
Себестоимость проду